# 01 · Raw CSV → typed silver Parquet
Reads the REES46 CSVs (Oct + Nov 2019) once, validates and de-duplicates them with the same rules as the Assignment 1 silver layer, and writes partitioned Parquet. Every later notebook reads from `data/silver/events`, never from the CSVs.

Feeds report §5.1 (Data processing). Metrics are saved to `ml/outputs/01_metrics.json`.

In [1]:
import os, json, time
from pyspark.sql import SparkSession, functions as F, types as T

DATA = "/home/jovyan/work/data"
SILVER = f"{DATA}/silver/events"
QUAR = f"{DATA}/silver/_quarantine"
OUT = "/home/jovyan/work/ml/outputs"
os.makedirs(f"{DATA}/spark-tmp", exist_ok=True)

spark = (SparkSession.builder.master("local[*]").appName("01_csv_to_silver")
         .config("spark.driver.memory", "12g")
         .config("spark.sql.shuffle.partitions", "64")
         .config("spark.local.dir", f"{DATA}/spark-tmp")
         .config("spark.sql.session.timeZone", "UTC")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print(spark.version, spark.sparkContext.defaultParallelism, "cores")

3.5.3 10 cores


## 1. Read raw CSV with an explicit schema

In [2]:
schema = T.StructType([
    T.StructField("event_time", T.StringType()),
    T.StructField("event_type", T.StringType()),
    T.StructField("product_id", T.LongType()),
    T.StructField("category_id", T.LongType()),
    T.StructField("category_code", T.StringType()),
    T.StructField("brand", T.StringType()),
    T.StructField("price", T.DoubleType()),
    T.StructField("user_id", T.LongType()),
    T.StructField("user_session", T.StringType()),
])
raw = spark.read.csv([f"{DATA}/2019-Oct.csv", f"{DATA}/2019-Nov.csv"], header=True, schema=schema)
raw.printSchema()

root
 |-- event_time: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: long (nullable = true)
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: long (nullable = true)
 |-- user_session: string (nullable = true)



## 2. Type, flag invalid rows, split `category_code`
Invalid = null/non-positive price, missing `user_id`/`user_session`, unparsable timestamp, or unknown `event_type`.

In [3]:
typed = (raw
    .withColumn("event_time", F.to_timestamp(F.regexp_replace("event_time", " UTC$", ""), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("event_date", F.to_date("event_time"))
    .withColumn("cat_parts", F.split("category_code", r"\."))
    .withColumn("cat_l1", F.col("cat_parts")[0])
    .withColumn("cat_l2", F.col("cat_parts")[1])
    .withColumn("cat_l3", F.col("cat_parts")[2])
    .drop("cat_parts")
    .withColumn("brand", F.lower(F.trim("brand")))
)
valid_cond = (
    F.col("event_time").isNotNull()
    & F.col("event_type").isin("view", "cart", "purchase")
    & (F.col("price") > 0)
    & F.col("user_id").isNotNull()
    & F.col("user_session").isNotNull() & (F.col("user_session") != "")
)
valid = typed.filter(valid_cond)
invalid = typed.filter(~valid_cond)

## 3. De-duplicate on the natural key and write Parquet
Same key as the streaming job: `(event_time, user_id, product_id, event_type, user_session)`.

In [4]:
t0 = time.time()
dedup = valid.dropDuplicates(["event_time", "user_id", "product_id", "event_type", "user_session"])
(dedup.repartition("event_date")
      .write.mode("overwrite").partitionBy("event_date").parquet(SILVER))
t_silver = time.time() - t0

t0 = time.time()
invalid.write.mode("overwrite").parquet(QUAR)
t_quar = time.time() - t0
print(f"silver written in {t_silver/60:.1f} min, quarantine in {t_quar/60:.1f} min")

silver written in 4.1 min, quarantine in 1.2 min


## 4. Verify and collect metrics

In [5]:
silver = spark.read.parquet(SILVER)
n_raw = raw.count()
n_silver = silver.count()
n_quar = spark.read.parquet(QUAR).count()
n_dup = n_raw - n_quar - n_silver
print(f"raw={n_raw:,}  silver={n_silver:,}  quarantine={n_quar:,}  duplicates_removed={n_dup:,}")
assert n_raw == n_silver + n_quar + n_dup

raw=109,950,743  silver=109,563,324  quarantine=256,773  duplicates_removed=130,646


In [6]:
funnel = (silver.groupBy("event_type").count().toPandas()
          .set_index("event_type")["count"].to_dict())
card = silver.agg(
    F.countDistinct("user_id").alias("users"),
    F.countDistinct("product_id").alias("products"),
    F.countDistinct("user_session").alias("sessions"),
    F.min("event_date").alias("first_day"),
    F.max("event_date").alias("last_day"),
    F.avg((F.col("brand").isNull()).cast("int")).alias("brand_null_share"),
    F.avg((F.col("category_code").isNull()).cast("int")).alias("cat_null_share"),
).first().asDict()
card = {k: (str(v) if k.endswith("_day") else v) for k, v in card.items()}
metrics = dict(raw=n_raw, silver=n_silver, quarantine=n_quar, duplicates=n_dup,
               funnel=funnel, cardinality=card,
               minutes_silver=round(t_silver/60, 2), minutes_quarantine=round(t_quar/60, 2))
json.dump(metrics, open(f"{OUT}/01_metrics.json", "w"), indent=2, default=str)
metrics

{'raw': 109950743,
 'silver': 109563324,
 'quarantine': 256773,
 'duplicates': 130646,
 'funnel': {'cart': 3823980, 'purchase': 1659703, 'view': 104079641},
 'cardinality': {'users': 5315274,
  'products': 205415,
  'sessions': 22995676,
  'first_day': '2019-10-01',
  'last_day': '2019-11-30',
  'brand_null_share': 0.137497142748243,
  'cat_null_share': 0.32173653292957777},
 'minutes_silver': 4.11,
 'minutes_quarantine': 1.15}

In [7]:
silver.groupBy("event_type").count().show()
silver.limit(5).toPandas()

+----------+---------+
|event_type|    count|
+----------+---------+
|      cart|  3823980|
|  purchase|  1659703|
|      view|104079641|
+----------+---------+



,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,cat_l1,cat_l2,cat_l3,event_date
0,2019-11-16 06:50:52,view,17800157,2053013559868129947,computers.desktop,aerocool,205.90,519782931,00011333-67e9-46bb-a0d1-54ff8540e2b9,computers,desktop,None,2019-11-16
1,2019-11-16 13:09:10,view,26500136,2053013563550729061,None,lucente,91.38,512448633,82d877a5-7186-4be9-b688-b7ea0a8acf30,None,None,None,2019-11-16
2,2019-11-16 06:54:37,view,19200103,2053013556202308035,construction.tools.saw,stihl,236.81,518902510,00021f98-dd43-43bb-869d-50445664674a,construction,tools,saw,2019-11-16
3,2019-11-16 05:13:05,view,22700127,2053013556168753601,None,stels,100.65,558543806,82d8e763-ff40-42ff-8788-3b70e89865b3,None,None,None,2019-11-16
4,2019-11-16 07:02:54,view,1305977,2053013558920217191,computers.notebook,apple,2548.07,545742684,000277a5-2b64-4cbb-a185-ebab071a143a,computers,notebook,None,2019-11-16


In [8]:
spark.stop()